# word2vec 作为矩阵分解

我们先从一个简单的训练集开始。


In [1]:
train = ['The student speaks, as the student wants to learn. We learn what the student wants.']

In [ ]:
# （以后再说）
# 网上有大量免费的书！
# 阿瑟·柯南·道尔的《福尔摩斯探案集》
# http://www.gutenberg.org/ebooks/1661
'''!wget http://www.gutenberg.org/files/1661/1661-0.txt
with open('1661-0.txt') as f:
    train = [f.read()]'''

In [ ]:
# （更晚以后）
# 更大的数据集，100 MB（1700 万词）
'''!wget http://mattmahoney.net/dc/text8.zip
!unzip text8.zip
with open('text8') as f:
    train = [f.read()]'''

In [2]:
len(train[0].split()), 'words'

(15, 'words')

首先，我们把文本标准化（转小写、去掉标点等）。


In [3]:
%%time
from sklearn.feature_extraction.text import CountVectorizer

transformer = CountVectorizer()
transformer.fit(train)
analyzer = transformer.build_analyzer()

tokens = analyzer(train[0])
print(tokens)

['the', 'student', 'speaks', 'as', 'the', 'student', 'wants', 'to', 'learn', 'we', 'learn', 'what', 'the', 'student', 'wants']
CPU times: user 1.12 s, sys: 998 ms, total: 2.12 s
Wall time: 391 ms


In [4]:
encoder = transformer.vocabulary_
encoder

{'the': 4,
 'student': 3,
 'speaks': 2,
 'as': 0,
 'wants': 6,
 'to': 5,
 'learn': 1,
 'we': 7,
 'what': 8}

In [5]:
decoder = transformer.get_feature_names()
decoder

['as', 'learn', 'speaks', 'student', 'the', 'to', 'wants', 'we', 'what']

上下文是语料中每个词周围大小为 `WINDOW_SIZE` 的窗口。

如果语料是 $w_0, \ldots, w_{n - 1}$，那么词 $w_i$ 的上下文就是所有词 $w_{i - L}, w_{i - L + 1}, \ldots, w_{i + L}$，其中 $L$ 就是 `WINDOW_SIZE`。

写一段代码，构建一个**词**-上下文共现计数矩阵。注意边界情况。

**The** student $\rightarrow$ *student* 是 ***The*** 的一个上下文，所以应该给这个词-上下文对加一  
The **student** speaks $\rightarrow$ *The* 和 *speaks* 是 ***student*** 的上下文  
student **speaks** as


In [ ]:
%%time
from collections import Counter

WINDOW_SIZE = 1  # 先取 1，更大的语料再取 5
counts = Counter()  # 保存词-上下文出现次数（键是二元组）
nb_word = Counter()  # 保存每个词的出现次数
nb_context = Counter()  # 保存每个上下文的出现次数

for pos, word in enumerate(tokens):
    # 在这里写你的代码
    pass
# 检查计数

现在我们来构建一个词-上下文 PMI 矩阵（*逐点互信息*），经验公式如下：

$$ PMI(w, c) = \log \frac{P(w, c)}{P(w)P(c)} = \log \frac{\#(w, c) |D|}{\#(w) \#(c)} $$

其中 $|D|$ 是语料中词-上下文对的数量，$\#(w), \#(c), \#(w, c)$ 分别是词、上下文和词-上下文对的出现次数。

这个矩阵是稀疏的：请填充 `rows`、`cols` 和 `data` 三个列表，分别对应词索引（用定义为词表的 `encoder`）、上下文索引和计数。


In [ ]:
%%time
import numpy as np

rows = []  # rows = []  # 词的索引
cols = []  # cols = []  # 上下文的索引
data = []  # data = []  # 矩阵的值

for (word, context), count in counts.items():
    # 在这里写你的代码
    pass

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

pmi = csr_matrix((data, (rows, cols)), shape=(len(nb_word), len(nb_context)))
pmi

我们将计算这个矩阵的 SVD 来降维。

一个 $m \times n$ 的秩为 $r$ 的矩阵 $M$ 的（紧凑）奇异值分解是 $U \Sigma V^T$，其中：

- $U$ 是半酉矩阵，大小为 $m \times r$
- $\Sigma$ 是对角矩阵，$r \times r$
- $V^T$ 是半酉矩阵，大小为 $r \times n$，即 $U^T U = V^T V = I_{r \times r}$。

我们通常按降序排列奇异值，以便解释尽可能多的方差（$k$-SVD 是秩为 $k$ 的最优近似）。


In [ ]:
%%time
u, sigma, vt = svds(pmi, k=3)
# 在 text8 上约 1 分 55 秒

In [ ]:
pmi.min(), pmi.max()

In [ ]:
embeddings = u * sigma
embeddings.shape

In [ ]:
u.shape, sigma.shape, vt.shape

## 词相似度


现在来玩一下词向量！

实现余弦相似度：

$$ cos(u, v) = \frac{\langle u, v \rangle}{|| u ||_2 || v ||_2} $$

然后验证你的结果和 `sklearn` 一致。

现在是时候转到 Sherlock Holmes 语料了。回到第一个单元格！

在词表（`encoder`）里挑几个词，计算它们的 20 个最近邻。


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# 在这里写你的代码

实际上，相似度值里有噪声。让我们把 PMI 矩阵里的负值去掉。

> *在表示词的时候，忽略负值背后有一些直觉：人类很容易想到正关联（比如"加拿大"和"雪"），但要编造负关联（"加拿大"和"沙漠"）就难得多。这说明两个词的感知相似度更多受它们共享的正上下文影响，而不是共享的负上下文。因此，把负关联的上下文丢弃、标记为"无信息"（0），在直觉上是说得通的。*

现在它是 PPMI 矩阵：正逐点互信息。


In [ ]:
ppmi = pmi.copy()
ppmi.data[ppmi.data < 0] = 0
ppmi.eliminate_zeros()  # 去掉现在为 0 的值
ppmi
# 用 ppmi 重新计算 SVD

## 语义类比


现在**让我们用代数代替逻辑。**

（我猜不是每个人都会对这个说法满意。）

为更大的数据集 `text8` 重新计算 PPMI 矩阵和词向量。

然后试着回答一些问题，我们需要在下面的式子里找 $b^*$：

$$ a \textrm{ is to } a^* \textrm{ as } b \textrm{ is to } b^* $$

（例如 *Paris is to France as Tokyo is to Japan*）

怎么用词向量来表达这个？


In [ ]:
# 等 text8 训练完之后
# encoder['paris'], encoder['france'], encoder['tokyo'], encoder['japan']
# a a* b (b*?)

它有效吗？你可能想归一化词向量。请把它们归一化到 `embed_unit`。

请找一找那些"刁钻"的类比（不公平的捷径）。


In [ ]:
# 在这里写你的代码

请注意，复用你的分解方法来学习词向量是可行的。这是以后一个作业的主题！

# 参考文献

值得一读！

Levy, O., & Goldberg, Y. (2014). [Neural word embedding as implicit matrix factorization.](https://papers.nips.cc/paper/5477-neural-word-embedding-as-implicit-matrix-factorization.pdf) In Advances in neural information processing systems (pp. 2177–2185).

Doyle, A. C. (1891). [The Adventures of Sherlock Holmes: Adventure I. — A Scandal in Bohemia.](http://www.gutenberg.org/ebooks/1661) The Strand Magazine, vol. 2, pp. 61–75 (July 1891).
